In [ ]:
import json
from website_summarizer import Model, SeleniumWebScrapper, ChatOptions
from IPython.display import Markdown, display, update_display

class BrochureGenerator:
    def __init__(self) -> None:
        # uses default OpenAI Model
        self.content_model = Model(model="gpt-5-nano")
        self.brochure_model = Model(model="gpt-4.1-mini")

        
    def _get_link_system_prompt(self) -> str:
        return """
            You are provided with a list of links found on a webpage.
            You are able to decide which of the links would be most relevant to include in a brochure about the company,
            such as links to an About page, or a Company page, or Careers/Jobs pages.
            You should respond in JSON as in this example:
            {
                "links": [
                    {"type": "about page", "url": "https://full.url/goes/here/about"},
                    {"type": "careers page", "url": "https://another.full.url/careers"}
                ]
            }
        """
    
    def _get_links_user_prompt(self, url: str) -> str:
        user_prompt = f""" 
            Here is the list of links on the website {url} -
            Please decide which of these are relevant web links for a brochure about the company, 
            respond with the full https URL in JSON format.
            Do not include Terms of Service, Privacy, email links.

            Links (some might be relative links):
        """
        scrapper = SeleniumWebScrapper(url)
        links = scrapper.fetch_website_links()
        print(len(links), links)
        user_prompt += "\n".join(links)

        return user_prompt

    def select_relevant_links(self, url: str):
        messages = [
            {"role": "system", "content": self._get_link_system_prompt()},
            {"role": "user", "content": self._get_links_user_prompt(url)}
        ]

        options = ChatOptions(response_format={"type": "json_object"}, stream=None)
        result = self.content_model.chat(messages=messages, options=options)
        links = json.loads(result)

        return links

    def fetch_page_and_all_relevant_links(self, url: str) -> str:
        scrapper = SeleniumWebScrapper(url)
        contents = scrapper.fetch_website_contents()
        relevant_links = self.select_relevant_links(url)

        result = f"## Landing Page: \n\n{contents}\n ## Relevant Links: \n"

        for link in relevant_links["links"]:
            link_scrapper = SeleniumWebScrapper(link['url'])
            result += f"\n\n ### Link: {link['type']}\n"
            result += link_scrapper.fetch_website_contents()
        
        return result


    def get_brochure_system_prompt(self) -> str:
        return """
            You are an assistant that analyzes the contents of several relevant pages from a company website
            and creates a short brochure about the company for prospective customers, investors and recruits.
            Respond in markdown without code blocks.
            Include details of company culture, customers and careers/jobs if you have the information.
        """

    def get_brochure_user_prompt(self, company_name, url) -> str:
        user_prompt = f""" 
            You are looking at a company called: {company_name}
            Here are the contents of its landing page and other relevant pages;
            use this information to build a short brochure of the company in markdown without code blocks.\n\n
        """

        user_prompt += self.fetch_page_and_all_relevant_links(url)
        user_prompt = user_prompt[:5_000] 
        return user_prompt

    def create_brochure(self, company_name, url, stream = False) -> str:
        messages = [
            {"role": "system", "content": self.get_brochure_system_prompt()},
            {"role": "user", "content": self.get_brochure_user_prompt(company_name, url)}
        ]

        options = ChatOptions(stream=stream, response_format=None)

        if stream:
            response = ""
            display_handle = display(Markdown(response), display_id=True)
            for chunk in self.brochure_model.chat_stream(messages=messages, options=options):
                response += chunk
                update_display(Markdown(response), display_id=display_handle.display_id)
        else:
            result = self.brochure_model.chat(messages=messages, options=options)
            display(Markdown(result))


In [2]:
bg = BrochureGenerator()

In [4]:
bg.create_brochure("HuggingFace", "https://huggingface.co", True)

['https://huggingface.co/', 'https://huggingface.co/models', 'https://huggingface.co/datasets', 'https://huggingface.co/spaces', 'https://huggingface.co/storage', 'https://huggingface.co/docs', 'https://huggingface.co/enterprise', 'https://huggingface.co/pricing', 'https://huggingface.co/tasks', 'https://huggingface.co/spaces', 'https://huggingface.co/models', 'https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash-0731', 'https://huggingface.co/MiniMaxAI/MiniMax-H3', 'https://huggingface.co/moonshotai/Kimi-K3', 'https://huggingface.co/Comfy-Org/MiniMax-H3', 'https://huggingface.co/DavidAU/Qwen3.6-27B-Fable-Fusion-711-Uncensored-Heretic-NM-DAU-NEO-MAX-MTP-GGUF', 'https://huggingface.co/models', 'https://huggingface.co/spaces/cinderholm/wan2-2-i2v-v3', 'https://huggingface.co/spaces/prithivMLmods/Qwen-Image-Edit-2511-LoRAs-Fast', 'https://huggingface.co/spaces/kulkas2pintu/wan555', 'https://huggingface.co/spaces/obsxrver/wan22-i2v-omni-lora', 'https://huggingface.co/spaces/cruisewagner222

# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future of machine learning. It serves as the premier platform where the global machine learning community collaborates on models, datasets, and applications across various AI modalities including text, image, video, audio, and even 3D.

With over 2 million models and 500,000+ datasets available, Hugging Face empowers AI developers, researchers, and enterprises to create, share, and discover cutting-edge machine learning tools in a highly collaborative, open environment.

---

## Our Platform & Offerings

- **Community-Driven Collaboration:** Host and collaborate on unlimited public models, datasets, and applications.
- **Extensive Model & Dataset Library:** Access and contribute to 2M+ machine learning models and 500k+ datasets.
- **Multi-Modal AI:** Work with AI across text, image, video, audio, and 3D formats.
- **Explore AI Applications:** Browse and interact with over 1 million AI applications built by the community.
- **Open Source Stack:** Accelerate your projects with Hugging Face's open source tools and frameworks.
- **Paid Compute & Enterprise Solutions:** Scale your AI development with advanced compute options and collaborative enterprise-grade services.

---

## Company Culture

Hugging Face thrives on a vibrant, open, and collaborative culture that fuels innovation:

- **Community at the Core:** Driven by the passion of AI researchers, engineers, and enthusiasts worldwide.
- **Transparency & Openness:** Committed to open source values and knowledge sharing.
- **Innovation Focus:** Encourages contributions to the latest AI research and real-world applications.
- **Building Together:** A platform that supports both individual creators and teams to build their machine learning portfolios and reputations.
- **Continuous Learning:** Active engagement through updates, datasets, and research papers fosters ongoing growth and expertise.

---

## Our Customers & Community

Hugging Face serves a diverse and global audience:

- **AI researchers and scientists** seeking collaborative environments and cutting-edge datasets.
- **Machine learning engineers and developers** building AI products and applications.
- **Enterprises and teams** needing scalable and collaborative AI infrastructure.
- **Educators and students** exploring state-of-the-art AI models and tools for learning and experimentation.
- **Open-source contributors** passionate about advancing AI technology together.

---

## Careers & Opportunities

Join Hugging Face and contribute to the future of AI!

- **Work with the latest AI research and technologies** in a supportive, open community.
- **Collaborate across disciplines** with top talent in AI and machine learning.
- **Help millions of developers** and enterprises build smarter AI applications worldwide.
- **Enjoy a culture of innovation, transparency, and continuous learning.**

Visit [huggingface.co/careers](https://huggingface.co) to explore current job openings and become part of the AI revolution.

---

## Get Started with Hugging Face

- Explore AI apps and models: [huggingface.co/models](https://huggingface.co/models)
- Browse datasets and applications: [huggingface.co/datasets](https://huggingface.co/datasets)
- Join the community and start collaborating: [sign up](https://huggingface.co/join)

**Hugging Face — The AI community building the future.**